# 19 (DE) — Federation via Foreign Tables

**Data Engineer perspective.** Query remote data without copying it: IRIS Foreign Tables map remote relations (JDBC sources or files) into local SQL. This notebook federates **this same IRIS instance** over JDBC so the mechanics run deterministically — in production the URL simply points at another server.

**Prerequisites**: the IRIS JDBC gateway (built-in) and reachability from the IRIS container to the target URL.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Local seed data

A small dimension we will federate back and join against.

In [ ]:
import pandas as pd

dim = session.createDataFrame(pd.DataFrame({
    "cliente_id": [1, 2, 3, 4, 5],
    "segmento": ["vip", "padrao", "padrao", "vip", "padrao"],
}))
local_fato = session.createDataFrame(pd.DataFrame({
    "pedido_id": [1, 2, 3],
    "cliente_id": [1, 2, 99],
    "valor": [100.0, 250.0, 90.0],
}))
local_fato.show()

## 2. Register a JDBC foreign table

The URL targets this same IRIS (`localhost:1972` inside the container). Swap the host/port for any remote IRIS or third-party database.

In [ ]:
URL = "jdbc:IRIS://localhost:1972/DATASPARK"
try:
    dim_ft = session.iris.register_jdbc_foreign_table(
        url=URL,
        dbtable="vendas",
        user="suser",
        password="pass123",
        driver="com.intersystems.jdbc.IRISDriver",
        name="ft_vendas_demo",
    )
    print("registered:", type(dim_ft).__name__)
except Exception as e:
    print("federation not available:", str(e)[:160])

## 3. Query the remote relation locally

Rows never land in Python until an action runs.

In [ ]:
try:
    dim_ft.select("*").show(3)
except NameError:
    print("skipped: foreign table was not registered")

## 4. Cross-source join

Local DataFrame joined to the federated table — one SQL statement, executed entirely inside IRIS. The remote `vendas` table has an `id` column that overlaps `local_fato.cliente_id`.

In [ ]:
try:
    relatorio = local_fato.join(dim_ft, local_fato.cliente_id == dim_ft.id, "left")
    relatorio.select("pedido_id", "estado").show()
except NameError:
    print("skipped: foreign table was not registered")

## 5. Write back through federation — `writer.jdbc`

Pushes the DataFrame's SQL result through the foreign server to a remote table. **Note**: IRIS's JDBC foreign-table write path has a known internal limitation (`copyFTInformation` error) when the source is a temp table, so this cell is best-effort.

In [ ]:
try:
    local_fato.write.jdbc(
        url=URL,
        dbtable="ft_write_target",
        user="suser",
        password="pass123",
        driver="com.intersystems.jdbc.IRISDriver",
        mode="overwrite",
    )
    print("write-back ok")
except Exception as e:
    print("jdbc write not available:", str(e)[:140])

## 6. File federation — `register_file_foreign_table`

IRIS can also own file access; the path must exist on the IRIS server filesystem. Here we read a CSV the notebook wrote into `data/` (mounted at `/irispark-data` in compose).

In [ ]:
import pandas as pd
import os

csv_dir = "data/foreign_demo"
os.makedirs(csv_dir, exist_ok=True)
rows = dim.collect()[:3]
pd.DataFrame([tuple(r) for r in rows], columns=dim.columns).to_csv(
    os.path.join(csv_dir, "dim.csv"), index=False
)

try:
    ft_file = session.iris.register_file_foreign_table(
        path=os.path.join(csv_dir, "dim.csv"),
        server_path="/irispark-data/foreign_demo",
        name="ft_eventos_demo",
        options={"header": True},
    )
    ft_file.show(3)
except Exception as e:
    print("file federation needs a server-side path:", str(e)[:140])

## 7. Cleanup

Transient registrations are dropped on `session.close()`; explicit drops keep repeat runs tidy.

In [ ]:
for t in ("ft_vendas_demo", "ft_eventos_demo", "ft_write_target"):
    try:
        session.sql(f'DROP TABLE IF EXISTS "{t}"')
    except Exception:
        pass
print("cleaned up")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")